# 04_New_Models_Comparison.ipynb

This notebook implements and evaluates two new alternative models to address the J-shaped distribution of star ratings:
1. **Ordinal Logistic Regression** (Proportional Odds Model)
2. **Binary Logistic Regression** (5-stars vs. 1-4 stars)

We use the same 5-fold cross-validation setup (random_state=42) and GLASSO-selected interactions as the existing linear models to ensure comparability.

In [3]:
import os
import sys
import pandas as pd
import numpy as np
import pickle
import ast

# Add parent directory to path for imports
sys.path.append(os.path.abspath('..'))

from src.modeling_alternative import run_ordinal_cv, run_binary_cv, get_existing_results
from src.modeling import prepare_raw_modeling_data

# 1. Load Local Data
data_path = '../data/Seminar_Amazon_Results_FULL.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"✅ Local data loaded: {len(df)} rows.")
    
    # Parse stringified lists of tuples if they are strings
    if isinstance(df['aspect_sentiments'].iloc[0], str):
        print("Parsing aspect_sentiments column...")
        df['aspect_sentiments'] = df['aspect_sentiments'].apply(ast.literal_eval)
else:
    print(f"❌ Error: File not found at {data_path}. Please ensure the CSV is in the './data/' folder.")

✅ Local data loaded: 701528 rows.
Parsing aspect_sentiments column...


## Part 1: Ordinal Logistic Regression
We treat star ratings as ordered categories.

In [ ]:
print("Running Ordinal Logistic Regression CV...")
ordinal_results = run_ordinal_cv(df)

# Save results
with open('../model_ordinal_results.pkl', 'wb') as f:
    pickle.dump(ordinal_results, f)

def summarize_ordinal(res_list):
    df_res = pd.DataFrame(res_list)
    return df_res.mean()

ord_add_summary = summarize_ordinal(ordinal_results['additive'])
ord_int_summary = summarize_ordinal(ordinal_results['interaction'])

print("\nOrdinal Additive Summary:")
print(ord_add_summary)
print("\nOrdinal Interaction Summary:")
print(ord_int_summary)

2026-05-31 22:22:09,574 - INFO - Pivoting raw sentiment features...


Running Ordinal Logistic Regression CV...


2026-05-31 22:22:10,704 - INFO - Sparsity Filter: Dropped 379395 empty reviews. Remaining valid reviews: 322133
2026-05-31 22:22:10,705 - INFO - Starting Ordinal Logistic Regression Cross-Validation...
2026-05-31 22:22:10,722 - INFO - Processing Fold 1/5...
2026-05-31 22:23:52,371 - INFO - Standardizing feature matrix for 7 active aspects (N=257706)...
2026-05-31 22:23:52,427 - INFO - Starting EBIC selection over 100 lambda values (gamma=0.5)...
2026-05-31 22:23:52,651 - INFO - Selected lambda: 0.001000
2026-05-31 22:23:52,654 - INFO - Generating interaction features from 20 network edges...
2026-05-31 22:23:52,696 - INFO - Generating interaction features from 20 network edges...
2026-05-31 22:37:15,879 - INFO - Processing Fold 2/5...
2026-05-31 22:39:13,555 - INFO - Standardizing feature matrix for 7 active aspects (N=257706)...
2026-05-31 22:39:13,614 - INFO - Starting EBIC selection over 100 lambda values (gamma=0.5)...
2026-05-31 22:39:13,782 - INFO - Selected lambda: 0.001000
2026

## Part 2: Binary Logistic Regression
We predict whether a review is a 5-star review (1) or not (0).

In [ ]:
print("Running Binary Logistic Regression CV...")
binary_results = run_binary_cv(df)

# Save results
with open('../model_binary_results.pkl', 'wb') as f:
    pickle.dump(binary_results, f)

def summarize_binary(res_list):
    df_res = pd.DataFrame(res_list)
    return df_res.mean()

bin_add_summary = summarize_binary(binary_results['additive'])
bin_int_summary = summarize_binary(binary_results['interaction'])

print("\nBinary Additive Summary:")
print(bin_add_summary)
print("\nBinary Interaction Summary:")
print(bin_int_summary)

2026-05-31 20:55:24,540 - INFO - Pivoting raw sentiment features...


Running Binary Logistic Regression CV...


2026-05-31 20:55:25,383 - INFO - Sparsity Filter: Dropped 379395 empty reviews. Remaining valid reviews: 322133
2026-05-31 20:55:25,385 - INFO - Starting Binary Logistic Regression Cross-Validation...
2026-05-31 20:55:25,395 - INFO - Processing Fold 1/5...
c:\Users\Nativ\Documents\seminar\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
2026-05-31 20:55:25,775 - INFO - Standardizing feature matrix for 7 active aspects (N=257706)...
2026-05-31 20:55:25,851 - INFO - Starting EBIC selection over 100 lambda values (gamma=0.5)...
2026-05-31 20:55:26,053 - INFO - Selected lambda: 0.001000
2026-05-31 20:55:26,055 - INFO - Generating interaction features from 11 network e


Binary Additive Summary:
accuracy    0.747191
roc_auc     0.795362
f1          0.794392
dtype: float64

Binary Interaction Summary:
accuracy    0.745810
roc_auc     0.794334
f1          0.794080
dtype: float64


## Part 3: Final Comparison Table
Consolidating all models.

In [ ]:
existing = get_existing_results()

summary_data = []

# Linear Models
summary_data.append({
    'Model': 'Linear Additive',
    'Type': 'Continuous',
    'Metric 1 (RMSE/Acc)': existing['Linear Additive']['Avg RMSE'],
    'Metric 2 (AdjR2/AUC)': existing['Linear Additive']['Avg Adj R2'],
    'Complexity (BIC/AIC)': existing['Linear Additive']['Full BIC']
})
summary_data.append({
    'Model': 'Linear Interaction',
    'Type': 'Continuous',
    'Metric 1 (RMSE/Acc)': existing['Linear Interaction']['Avg RMSE'],
    'Metric 2 (AdjR2/AUC)': existing['Linear Interaction']['Avg Adj R2'],
    'Complexity (BIC/AIC)': existing['Linear Interaction']['Full BIC']
})

# Ordinal Models
summary_data.append({
    'Model': 'Ordinal Additive',
    'Type': 'Categorical',
    'Metric 1 (RMSE/Acc)': ord_add_summary['rps'], # Using RPS for Metric 1
    'Metric 2 (AdjR2/AUC)': ord_add_summary['log_likelihood'],
    'Complexity (BIC/AIC)': ord_add_summary['aic']
})
summary_data.append({
    'Model': 'Ordinal Interaction',
    'Type': 'Categorical',
    'Metric 1 (RMSE/Acc)': ord_int_summary['rps'],
    'Metric 2 (AdjR2/AUC)': ord_int_summary['log_likelihood'],
    'Complexity (BIC/AIC)': ord_int_summary['aic']
})

# Binary Models
summary_data.append({
    'Model': 'Binary Additive',
    'Type': 'Binary',
    'Metric 1 (RMSE/Acc)': bin_add_summary['accuracy'],
    'Metric 2 (AdjR2/AUC)': bin_add_summary['roc_auc'],
    'Complexity (BIC/AIC)': bin_add_summary['f1']
})
summary_data.append({
    'Model': 'Binary Interaction',
    'Type': 'Binary',
    'Metric 1 (RMSE/Acc)': bin_int_summary['accuracy'],
    'Metric 2 (AdjR2/AUC)': bin_int_summary['roc_auc'],
    'Complexity (BIC/AIC)': bin_int_summary['f1']
})

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*90)
print("FINAL MODEL COMPARISON SUMMARY")
print("="*90)
print(summary_df.to_string(index=False))
print("="*90)

print("\nNote: Metrics vary by model type. Linear uses RMSE/AdjR2/BIC. Ordinal uses RPS/LL/AIC. Binary uses Acc/AUC/F1.")


FINAL MODEL COMPARISON SUMMARY
              Model        Type  Metric 1 (RMSE/Acc)  Metric 2 (AdjR2/AUC)  Complexity (BIC/AIC)
    Linear Additive  Continuous             1.214400              0.335100          1.039391e+06
 Linear Interaction  Continuous             1.203400              0.347000          1.032536e+06
   Ordinal Additive Categorical             0.131469        -270355.407622          5.407328e+05
Ordinal Interaction Categorical             0.131446        -270248.686895          5.405414e+05
    Binary Additive      Binary             0.747191              0.795362          7.943923e-01
 Binary Interaction      Binary             0.745810              0.794334          7.940802e-01

Note: Metrics vary by model type. Linear uses RMSE/AdjR2/BIC. Ordinal uses RPS/LL/AIC. Binary uses Acc/AUC/F1.
